# Guía: preparar los datos de farmacias para InfluxDB (Jupyter)

**Objetivo (Tarea 1):** cargar el Excel, transformarlo y dejarlo listo para guardarlo en InfluxDB.

Recuerda cómo guarda los datos InfluxDB: cada fila necesita una **fecha** (`time`), unas **etiquetas** (tags, texto para filtrar) y unos **valores** (fields, números).

| Columna | Tipo en InfluxDB | Tabla |
|---|---|---|
| Fecha | time | todas |
| Country, Category, PromoFlag | tags | básicas (diaria y mensual) |
| PharmacyType, Brand, ProductID, ProductName, IsGeneric | tags | **detalle diaria** (nueva) |
| ventas, unidades, ingresos, costes, margen | fields | todas |
| ingresos_lista_eur, descuento_eur | fields | **detalle diaria** (nueva) |

**Pasos:** 1) cargar y unir todas las hojas, 2) crear la fecha, 3) sumar por día (básica y detalle), 4) sumar por mes, 5) comprobar, 6) guardar CSV, 7) subir a InfluxDB (opcional).

## Paso 1. Cargar las hojas del Excel y unirlas


In [17]:
import pandas as pd
import numpy as np

RUTA = "../Data/Pharmacy_data.xlsx"  # ruta relativa a este notebook (BDA/)

Datos = pd.read_excel(RUTA, sheet_name=None)
for nombre, df in Datos.items():
    print(nombre, df.shape)

FactSales (62139, 9)
DimDate (731, 7)
DimPharmacy (120, 10)
DimProduct (220, 11)


In [18]:
fact = Datos["FactSales"]
farmacia = Datos["DimPharmacy"]
producto = Datos["DimProduct"]

data = (
    fact
    .merge(farmacia[["PharmacyID", "Country", "Region", "PharmacyType"]],
           on="PharmacyID", how="left", validate="many_to_one")
    .merge(producto[["ProductID", "ProductName", "Category", "Brand", "IsGeneric", "ListPriceEUR"]],
           on="ProductID", how="left", validate="many_to_one")
)
data.head()

,SalesID,DateKey,PharmacyID,ProductID,UnitsSold,RevenueEUR,CostEUR,MarginEUR,PromoFlag,Country,Region,PharmacyType,ProductName,Category,Brand,IsGeneric,ListPriceEUR
0,S0000001,20240101,PH0002,PR0099,5,128.08,87.55,40.53,No,Austria,Tyrol,Urban,HairEssence Mouthwash Sensitive,Personal Care,HairEssence,No,24.63
1,S0000002,20240101,PH0004,PR0156,4,51.89,34.32,17.57,No,Poland,Silesian,Suburban,NatureFit Omega-3 Active,Wellness,NatureFit,No,16.63
2,S0000003,20240101,PH0007,PR0004,20,317.73,199.53,118.20,No,Germany,North Rhine-Westphalia,Rural,DermaSoft Shampoo Repair,Personal Care,DermaSoft,No,15.13
3,S0000004,20240101,PH0009,PR0075,6,90.34,67.49,22.85,No,Germany,Bavaria,Urban,Medica Paracetamol 20 mg,OTC,Medica,No,14.34
4,S0000005,20240101,PH0010,PR0154,2,160.21,124.99,35.22,Yes,Belgium,Wallonia,Urban,CareEquip Digital Thermometer Advance,Medical Devices,CareEquip,No,91.50


## Paso 2. Crear la columna de fecha

`DateKey` viene como número (por ejemplo `20240315`). InfluxDB necesita una fecha de verdad, y mejor con zona horaria (usamos UTC).

In [19]:
data["Fecha"] = pd.to_datetime(data["DateKey"].astype(str), format="%Y%m%d").dt.tz_localize("UTC")
data[["DateKey", "Fecha"]].head()

,DateKey,Fecha
0,20240101,2024-01-01 00:00:00+00:00
1,20240101,2024-01-01 00:00:00+00:00
2,20240101,2024-01-01 00:00:00+00:00
3,20240101,2024-01-01 00:00:00+00:00
4,20240101,2024-01-01 00:00:00+00:00


## Paso 3. Sumar las ventas por día

**Esto es lo más importante.** En InfluxDB, si dos filas tienen el mismo measurement, las mismas tags y la misma fecha, la segunda **sobrescribe** a la primera. Tus ventas se repiten mucho en un mismo día, país, categoría y promoción, así que las sumamos para que cada combinación sea única.

In [20]:
columnas_tag = ["Country", "Category", "PromoFlag"]

diario = (
    data.groupby(["Fecha"] + columnas_tag, as_index=False)
    .agg(ventas=("SalesID", "count"),
         unidades=("UnitsSold", "sum"),
         ingresos_eur=("RevenueEUR", "sum"),
         costes_eur=("CostEUR", "sum"),
         margen_eur=("MarginEUR", "sum"))
)
print(diario.shape)
diario.head()

(29435, 9)


,Fecha,Country,Category,PromoFlag,ventas,unidades,ingresos_eur,costes_eur,margen_eur
0,2024-01-01 00:00:00+00:00,Austria,Medical Devices,No,1,1,24.78,17.38,7.40
1,2024-01-01 00:00:00+00:00,Austria,OTC,No,5,58,522.57,378.83,143.74
2,2024-01-01 00:00:00+00:00,Austria,Personal Care,No,1,5,128.08,87.55,40.53
3,2024-01-01 00:00:00+00:00,Austria,Prescription,Yes,1,3,36.09,30.82,5.27
4,2024-01-01 00:00:00+00:00,Austria,Wellness,No,4,37,560.83,352.36,208.47


## Paso 3b. Ventas diarias con más detalle (producto, marca, tipo de farmacia...)

La tabla del Paso 3 solo distingue país, categoría y promoción. Esta añade **producto, marca, genérico y tipo de farmacia** como tags, y dos fields más:

- `ingresos_lista_eur`: lo que se habría ingresado al **precio de lista** (`UnitsSold × ListPriceEUR`).
- `descuento_eur`: `ingresos_lista_eur − ingresos_eur`. Si sale negativo, se vendió por encima del precio de lista.

**Cosas a tener en cuenta:**

- **Agrupamos por `ProductID`**, no solo por nombre: hay nombres de producto repetidos en productos distintos. El nombre va como tag extra para poder filtrar en Grafana.
- **Más tags = más series y menos agrupación.** Con estos tags salen unas 8.000 series y casi tantas filas como ventas, que InfluxDB aguanta bien. Si añades `Region` (38 valores) suben a casi 20.000. Cambia la lista `tags_detalle` para probar.
- **Solo guardamos sumas.** Porcentajes y medias (margen %, precio medio) se calculan después en Grafana como `sum(margen) / sum(ingresos)`; si los guardaras ya calculados y luego los promediaras, el resultado saldría mal.

In [21]:
tags_detalle = ["Country", "PharmacyType", "Category", "Brand",
                "ProductID", "ProductName", "IsGeneric", "PromoFlag"]

data["ingresos_lista_eur"] = data["UnitsSold"] * data["ListPriceEUR"]

diario_detalle = (
    data.groupby(["Fecha"] + tags_detalle, as_index=False)
    .agg(ventas=("SalesID", "count"),
         unidades=("UnitsSold", "sum"),
         ingresos_eur=("RevenueEUR", "sum"),
         costes_eur=("CostEUR", "sum"),
         margen_eur=("MarginEUR", "sum"),
         ingresos_lista_eur=("ingresos_lista_eur", "sum"))
)
diario_detalle["descuento_eur"] = diario_detalle["ingresos_lista_eur"] - diario_detalle["ingresos_eur"]

cols_eur = ["ingresos_eur", "costes_eur", "margen_eur", "ingresos_lista_eur", "descuento_eur"]
diario_detalle[cols_eur] = diario_detalle[cols_eur].round(2)

print(diario_detalle.shape, "| series:", diario_detalle.groupby(tags_detalle).ngroups)
diario_detalle.head()

(61566, 16) | series: 8106


,Fecha,Country,PharmacyType,Category,Brand,ProductID,ProductName,IsGeneric,PromoFlag,ventas,unidades,ingresos_eur,costes_eur,margen_eur,ingresos_lista_eur,descuento_eur
0,2024-01-01 00:00:00+00:00,Austria,Suburban,Medical Devices,HomeCheck,PR0022,HomeCheck Blood Pressure Monitor Compact,No,No,1,1,24.78,17.38,7.40,23.83,-0.95
1,2024-01-01 00:00:00+00:00,Austria,Suburban,OTC,ColdAway,PR0038,ColdAway Paracetamol 200 mg,Yes,No,1,6,35.44,24.82,10.62,34.08,-1.36
2,2024-01-01 00:00:00+00:00,Austria,Suburban,OTC,Medica,PR0068,Medica Throat Lozenges 500 mg,No,No,1,17,215.52,163.72,51.80,207.23,-8.29
3,2024-01-01 00:00:00+00:00,Austria,Suburban,OTC,PainLess,PR0113,PainLess Paracetamol 20 mg,No,No,1,18,126.92,87.93,38.99,122.04,-4.88
4,2024-01-01 00:00:00+00:00,Austria,Suburban,Wellness,ImmunoPlus,PR0047,ImmunoPlus Omega-3 Pure,No,No,1,13,190.50,119.31,71.19,183.17,-7.33


## Paso 4. Sumar las ventas por mes

Es el mismo cálculo, pero la fecha es el primer día de cada mes. Irá al segundo bucket (`ventas_mensuales`).

In [22]:
data["Mes"] = data["Fecha"].dt.tz_localize(None).dt.to_period("M").dt.to_timestamp().dt.tz_localize("UTC")

mensual = (
    data.groupby(["Mes"] + columnas_tag, as_index=False)
    .agg(ventas=("SalesID", "count"),
         unidades=("UnitsSold", "sum"),
         ingresos_eur=("RevenueEUR", "sum"),
         costes_eur=("CostEUR", "sum"),
         margen_eur=("MarginEUR", "sum"))
    .rename(columns={"Mes": "Fecha"})
)
print(mensual.shape)
mensual.head()

(1908, 9)


,Fecha,Country,Category,PromoFlag,ventas,unidades,ingresos_eur,costes_eur,margen_eur
0,2024-01-01 00:00:00+00:00,Austria,Medical Devices,No,12,34,"2,118.42","1,580.41",538.01
1,2024-01-01 00:00:00+00:00,Austria,Medical Devices,Yes,1,1,52.42,49.15,3.27
2,2024-01-01 00:00:00+00:00,Austria,OTC,No,64,576,"6,653.19","4,727.64","1,925.55"
3,2024-01-01 00:00:00+00:00,Austria,OTC,Yes,11,114,"1,023.63",790.54,233.09
4,2024-01-01 00:00:00+00:00,Austria,Personal Care,No,28,227,"3,614.48","2,375.84","1,238.64"


## Paso 4b (opcional). Datos simulados de 2026 con anomalías

Copia las ventas diarias de 2025 al año 2026, les añade ruido y dos anomalías (una racha de 16 días con -40 % de ventas y una bajada de precio de 10 días) para practicar alertas en Grafana. Usa la tabla básica `diario`, y se guarda en `ventas_simuladas.csv`.

In [23]:
rng = np.random.default_rng(42)
cols_vol = ["ventas", "unidades", "ingresos_eur", "costes_eur"]

sim = diario[diario["Fecha"].dt.year == 2025].copy()
sim["Fecha"] = sim["Fecha"] + pd.DateOffset(years=1)

# 1. Ruido coherente: el mismo factor para las cuatro columnas de cada fila
factor = 1 + rng.normal(0, 0.10, len(sim))
sim[cols_vol] = sim[cols_vol].mul(factor, axis=0)

# 2. Anomalía de volumen: mala racha de 16 días con -40 % de ventas
racha = (sim["Fecha"] >= pd.Timestamp("2026-09-10", tz="UTC")) & \
        (sim["Fecha"] <= pd.Timestamp("2026-09-25", tz="UTC"))
sim.loc[racha, cols_vol] *= 0.6

# 3. Anomalía de margen: bajada de precio de 10 días (ingresos -15 %, mismo coste)
precio = (sim["Fecha"] >= pd.Timestamp("2026-08-01", tz="UTC")) & \
         (sim["Fecha"] <= pd.Timestamp("2026-08-10", tz="UTC"))
sim.loc[precio, "ingresos_eur"] *= 0.85

# 4. Limpieza
sim["ventas"] = sim["ventas"].round().clip(lower=1).astype(int)
sim["unidades"] = np.maximum(sim["unidades"].round().astype(int), sim["ventas"])
sim["ingresos_eur"] = sim["ingresos_eur"].round(2)
sim["costes_eur"] = sim["costes_eur"].round(2)
sim["margen_eur"] = (sim["ingresos_eur"] - sim["costes_eur"]).round(2)

sim.to_csv("ventas_simuladas.csv", index=False)
print(sim.shape, "| filas con margen negativo:", (sim["margen_eur"] < 0).sum())

(14807, 9) | filas con margen negativo: 16


## Paso 5. Comprobar que no se ha perdido nada

Tres comprobaciones: que no hay filas repetidas (para que InfluxDB no sobrescriba), que los totales coinciden con el original, y que no hay nulos.

In [24]:
clave = ["Fecha"] + columnas_tag
clave_detalle = ["Fecha"] + tags_detalle
print("Duplicados diario        :", diario.duplicated(clave).sum())
print("Duplicados mensual       :", mensual.duplicated(clave).sum())
print("Duplicados diario detalle:", diario_detalle.duplicated(clave_detalle).sum())

print()
print("Ingresos originales      :", round(fact["RevenueEUR"].sum(), 2))
print("Ingresos diario          :", round(diario["ingresos_eur"].sum(), 2))
print("Ingresos mensual         :", round(mensual["ingresos_eur"].sum(), 2))
print("Ingresos diario detalle  :", round(diario_detalle["ingresos_eur"].sum(), 2))

print()
print("Nulos diario        :", diario.isna().sum().sum())
print("Nulos mensual       :", mensual.isna().sum().sum())
print("Nulos diario detalle:", diario_detalle.isna().sum().sum())

print()
print("Ventas originales vs detalle:", len(fact), "|", diario_detalle["ventas"].sum())

Duplicados diario        : 0
Duplicados mensual       : 0
Duplicados diario detalle: 0

Ingresos originales      : 8633977.31
Ingresos diario          : 8633977.31
Ingresos mensual         : 8633977.31
Ingresos diario detalle  : 8633977.31

Nulos diario        : 0
Nulos mensual       : 0
Nulos diario detalle: 0

Ventas originales vs detalle: 62139 | 62139


**Cómo leerlo:** los duplicados deben ser `0`, los totales de ingresos deben ser iguales (los cuatro) y los nulos `0`. Si algo no cuadra, no sigas: revisa los pasos anteriores.

## Paso 6. Guardar los datos en CSV

Esto sirve si luego la subida se hace con Node-RED (Tarea 2), que leería estos archivos.

In [25]:
diario.to_csv("ventas_diarias.csv", index=False)
mensual.to_csv("ventas_mensuales.csv", index=False)
diario_detalle.to_csv("ventas_diarias_detalle.csv", index=False)